# 17 - External Statistical Significance Tests

Bu notebook mevcut **S&P 500** ve **Reuters** prediction dosyalarında dört ana eşleştirilmiş karşılaştırmayı çalıştırır:

- S&P 500: RoBERTa vs Fine-tuned FinBERT
- S&P 500: Original FinBERT vs Fine-tuned FinBERT
- Reuters: Original FinBERT vs Fine-tuned FinBERT
- Reuters: RoBERTa vs Fine-tuned FinBERT

Testler:
- Exact McNemar
- 4 karşılaştırma üzerinde Holm düzeltmesi
- 5.000 tekrarlı paired bootstrap
- Macro-F1 farkı için %95 güven aralığı

Yeni eğitim yapılmaz; yalnızca kayıtlı tahminler kullanılır.


In [1]:
from pathlib import Path
import math, re, json, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")

try:
    from thesis_utils import PROJECT_ROOT
    PROJECT_DIR = Path(PROJECT_ROOT)
except Exception:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    PROJECT_DIR = next(
        (p for p in candidates if (p / "app").exists() and (p / "outputs").exists()),
        here,
    )

RESULTS_DIR = PROJECT_DIR / "results" / "external_statistical_significance"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["negative", "neutral", "positive"]
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 2026
ALPHA = 0.05

print("PROJECT_DIR:", PROJECT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)


PROJECT_DIR: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis
RESULTS_DIR: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\results\external_statistical_significance


In [2]:
def norm(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

CSV_FILES = [p for p in PROJECT_DIR.rglob("*.csv") if p.is_file()]
print("CSV sayisi:", len(CSV_FILES))

# Diagnostics only. The actual inputs are pinned in the next cell because the
# old score-based auto picker selected internal validation CSVs for RoBERTa and
# Original FinBERT instead of the external-test prediction files.
def read_head(p):
    try:
        return pd.read_csv(p, nrows=3, encoding="utf-8-sig")
    except Exception:
        try:
            return pd.read_csv(p, nrows=3)
        except Exception:
            return None

def has_prediction_signal(p):
    df = read_head(p)
    if df is None:
        return False
    cols = {norm(c) for c in df.columns}
    return bool(cols & {
        "prediction", "pred_label", "predicted_label", "prediction_id", "pred",
        "y_pred", "finbert_label", "finbert_label_clean",
        "roberta_base_nli_zero_shot_pred_label",
    })

for p in CSV_FILES:
    s = norm(str(p.relative_to(PROJECT_DIR)))
    if has_prediction_signal(p) and any(x in s for x in ["external", "zero_shot_sp500", "reuters_annotation_all_clean"]):
        print(p.relative_to(PROJECT_DIR))


CSV sayısı: 388

 sp500_ft
48 | D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\external_evaluation\sp500_external\predictions.csv
38 | D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\external_evaluation\reuters_external\predictions.csv
38 | D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\external_evaluation\synthetic_external\predictions.csv
28 | D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\results\val_predictions.csv
28 | D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\results\test_predictions.c

In [3]:
# Pinned external-test inputs. Set path=None only when that prediction file has
# not been generated yet; the related comparison will be skipped with a note.
SPECS = {
    "sp500_ft": {
        "path": PROJECT_DIR / "checkpoints" / "financial_sentiment_multi_model" / "finbert_target_finetuned_seed42" / "external_evaluation" / "sp500_external" / "predictions.csv",
        "pred_cols": ["prediction", "prediction_id"],
    },
    "sp500_roberta": {
        "path": PROJECT_DIR / "outputs" / "zero_shot_sp500_external_test_model_family" / "roberta_base_nli_zero_shot_predictions.csv",
        "pred_cols": ["roberta_base_nli_zero_shot_pred_label"],
    },
    "sp500_original": {
        "path": PROJECT_DIR / "outputs" / "zero_shot_sp500_external_test_model_family" / "roberta_base_nli_zero_shot_predictions.csv",
        "pred_cols": ["finbert_label_clean", "finbert_label"],
    },
    "reuters_ft": {
        "path": PROJECT_DIR / "checkpoints" / "financial_sentiment_multi_model" / "finbert_target_finetuned_seed42" / "external_evaluation" / "reuters_external" / "predictions.csv",
        "pred_cols": ["prediction", "prediction_id"],
    },
    "reuters_original": {
        "path": PROJECT_DIR / "db" / "annotations" / "reuters_5000" / "REUTERS_annotation_all_clean.csv",
        "pred_cols": ["finbert_label"],
        "gold_cols": ["final_label", "chatgpt_label"],
    },
    "reuters_roberta": {
        "path": None,
        "pred_cols": ["roberta_base_nli_zero_shot_pred_label", "pred_label", "prediction"],
        "missing_reason": "Reuters RoBERTa external prediction CSV is not present in this repository.",
    },
}

PATHS = {k: v["path"] for k, v in SPECS.items()}
missing_inputs = []

print("SECILEN DOSYALAR")
for k, spec in SPECS.items():
    p = spec["path"]
    if p is None:
        missing_inputs.append({"key": k, "reason": spec.get("missing_reason", "path is None")})
        print(f"{k:18s} -> SKIP | {spec.get('missing_reason', 'path is None')}")
    elif not Path(p).exists():
        missing_inputs.append({"key": k, "path": str(p), "reason": "file not found"})
        print(f"{k:18s} -> MISSING | {p}")
    else:
        print(f"{k:18s} -> {p}")


SEÇİLEN DOSYALAR
sp500_ft           -> D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\external_evaluation\sp500_external\predictions.csv
sp500_roberta      -> D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\multiseed\roberta_seed123\results\val_predictions.csv
sp500_original     -> D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\multiseed\finbert_seed123\results\val_predictions.csv
reuters_ft         -> D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\external_evaluation\reuters_external\predictions.csv
reuters_roberta    -> D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\multiseed\roberta_seed

In [4]:
def clean(s):
    return (s.astype("string").str.strip()
            .replace(["","nan","NaN","None","none","<NA>","<na>"], pd.NA))

def label(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    return {
        "negative":"negative","neg":"negative","0":"negative", "-1":"negative",
        "neutral":"neutral","neu":"neutral","1":"neutral", "0.0":"neutral",
        "positive":"positive","pos":"positive","2":"positive", "1.0":"positive",
    }.get(s, pd.NA)

def col(df, names):
    m = {norm(c): c for c in df.columns}
    for n in names:
        if n is None:
            continue
        if norm(n) in m:
            return m[norm(n)]
    return None

def first_label(df, names, required_name):
    out = pd.Series(pd.NA, index=df.index, dtype="object")
    used = []
    for name in names:
        c = col(df, [name])
        if c is None:
            continue
        used.append(c)
        out = out.fillna(df[c].map(label))
    if not used:
        raise ValueError(f"{required_name} kolonu yok: {df.columns.tolist()}")
    return out, used

def normalize_text(s):
    return (
        clean(s)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"[??]", '"', regex=True)
        .str.replace(r"[?]", "'", regex=True)
        .str.strip()
    )

def load_predictions(key, model, dataset):
    spec = SPECS[key]
    path = spec["path"]
    if path is None or not Path(path).exists():
        print(f"{dataset:8s} | {model:20s} | SKIP - prediction file missing")
        return None

    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path)

    gold_names = spec.get("gold_cols") or ["gold_label", "true_label", "label", "final_label", "chatgpt_label", "y_true", "target", "gold_id", "label_id", "true_id", "y_true_id"]
    pred_names = spec.get("pred_cols") or ["prediction", "pred_label", "predicted_label", "pred", "y_pred", "prediction_id", "pred_id", "predicted_id"]

    gold, used_gold = first_label(df, gold_names, f"{model}: gold label")
    pred_label, used_pred = first_label(df, pred_names, f"{model}: prediction")

    out = pd.DataFrame(index=df.index)
    out["gold"] = gold
    out["pred"] = pred_label

    # Keep all possible keys; align() will choose the strongest common key.
    a = col(df, ["annotation_id"])
    s = col(df, ["sample_id","id"])
    t = col(df, ["eval_text","text_en","text","headline","title","source_text"])

    out["annotation_key"] = clean(df[a]).str.lower() if a is not None else pd.NA
    out["sample_key"] = clean(df[s]).str.lower() if s is not None else pd.NA
    out["text_key"] = normalize_text(df[t]) if t is not None else pd.NA

    out = out[
        out["gold"].isin(LABELS)
        & out["pred"].isin(LABELS)
    ].copy().reset_index(drop=True)

    print(
        f"{dataset:8s} | {model:20s} | N={len(out)} | "
        f"gold={used_gold} | pred={used_pred} | "
        f"annotation={out['annotation_key'].notna().sum()} | "
        f"sample={out['sample_key'].notna().sum()} | "
        f"text={out['text_key'].notna().sum()}"
    )

    return out

pred = {
    "sp500_ft": load_predictions("sp500_ft", "Fine-tuned FinBERT", "S&P500"),
    "sp500_roberta": load_predictions("sp500_roberta", "RoBERTa", "S&P500"),
    "sp500_original": load_predictions("sp500_original", "Original FinBERT", "S&P500"),
    "reuters_ft": load_predictions("reuters_ft", "Fine-tuned FinBERT", "Reuters"),
    "reuters_original": load_predictions("reuters_original", "Original FinBERT", "Reuters"),
    "reuters_roberta": load_predictions("reuters_roberta", "RoBERTa", "Reuters"),
}


S&P500   | Fine-tuned FinBERT   | N=1060 | annotation=1060 | sample=1060 | text=1060
S&P500   | RoBERTa              | N=1432 | annotation=0 | sample=1432 | text=1432
S&P500   | Original FinBERT     | N=1432 | annotation=0 | sample=1432 | text=1432
Reuters  | Fine-tuned FinBERT   | N=5000 | annotation=5000 | sample=5000 | text=5000
Reuters  | RoBERTa              | N=1432 | annotation=0 | sample=1432 | text=1432
Reuters  | Original FinBERT     | N=1432 | annotation=0 | sample=1432 | text=1432


In [5]:
def _prepare_key(df, key_col):
    x = df[[key_col, "gold", "pred"]].copy()
    x = x[x[key_col].notna()].copy()

    # Do not use non-unique keys for paired tests.
    if x[key_col].duplicated().any():
        return None

    return x

def align(a, b, name_a, name_b, expected_n):
    """
    Choose the strongest common key automatically:
    annotation_id -> sample_id -> normalized text.
    """
    key_candidates = ["annotation_key", "sample_key", "text_key"]
    diagnostics = []

    for key_col in key_candidates:
        xa = _prepare_key(a, key_col)
        xb = _prepare_key(b, key_col)

        if xa is None or xb is None:
            diagnostics.append((key_col, "duplicate key"))
            continue

        if len(xa) == 0 or len(xb) == 0:
            diagnostics.append((key_col, 0))
            continue

        left = xa.rename(columns={"gold":"gold_a","pred":"pred_a"})
        right = xb.rename(columns={"gold":"gold_b","pred":"pred_b"})

        m = left.merge(right, on=key_col, how="inner", validate="one_to_one")
        diagnostics.append((key_col, len(m)))

        if len(m) == expected_n:
            gold_mismatch = int((m["gold_a"] != m["gold_b"]).sum())
            if gold_mismatch:
                raise ValueError(
                    f"{name_a} vs {name_b}: {key_col} matched {expected_n} rows "
                    f"but {gold_mismatch} gold labels differ."
                )

            print(
                f"{name_a} vs {name_b}: {expected_n}/{expected_n} rows "
                f"matched by '{key_col}'."
            )

            return pd.DataFrame({
                "match_key": m[key_col],
                "gold": m["gold_a"],
                "pred_a": m["pred_a"],
                "pred_b": m["pred_b"],
            })

    diag_text = ", ".join(f"{k}={v}" for k, v in diagnostics)
    raise ValueError(
        f"{name_a} vs {name_b}: exact pairing failed. "
        f"Expected N={expected_n}. Tried keys: {diag_text}. "
        "Check that both prediction files are from the same external test set."
    )

comparison_specs = [
    ("SP500_RoBERTa_vs_FTFinBERT", "sp500_roberta", "sp500_ft", "RoBERTa", "FT FinBERT", 1060),
    ("SP500_OriginalFinBERT_vs_FTFinBERT", "sp500_original", "sp500_ft", "Original FinBERT", "FT FinBERT", 1060),
    ("Reuters_OriginalFinBERT_vs_FTFinBERT", "reuters_original", "reuters_ft", "Original FinBERT", "FT FinBERT", 5000),
    ("Reuters_RoBERTa_vs_FTFinBERT", "reuters_roberta", "reuters_ft", "RoBERTa", "FT FinBERT", 5000),
]

pairs = {}
skipped_comparisons = []
for cid, key_a, key_b, name_a, name_b, expected_n in comparison_specs:
    if pred.get(key_a) is None or pred.get(key_b) is None:
        skipped_comparisons.append({
            "comparison_id": cid,
            "reason": f"missing input: {key_a if pred.get(key_a) is None else key_b}",
        })
        print("SKIP", cid, "-", skipped_comparisons[-1]["reason"])
        continue
    pairs[cid] = align(pred[key_a], pred[key_b], name_a, name_b, expected_n)

for k, v in pairs.items():
    print(k, "N=", len(v))


ValueError: RoBERTa vs FT FinBERT: tam eşleşme bulunamadı. Beklenen N=1060. Denenen anahtarlar: annotation_key=0, sample_key=0, text_key=0. Prediction dosyalarının aynı test setinden olduğundan ve metin kolonlarının kaydedildiğinden emin ol.

In [ ]:
def binom_exact_two_sided(k, n):
    if n == 0:
        return 1.0
    try:
        from scipy.stats import binomtest
        return float(binomtest(int(k), int(n), p=0.5, alternative="two-sided").pvalue)
    except Exception:
        # Exact two-sided binomial p-value without materializing huge comb(n, i)
        # values as floats. This fallback is slower but numerically safe enough for
        # the dataset sizes used here.
        k = int(k)
        n = int(n)
        p0 = 0.5 ** n
        probs = [p0]
        for i in range(0, n):
            probs.append(probs[-1] * (n - i) / (i + 1))
        obs = probs[k]
        return float(min(1.0, sum(p for p in probs if p <= obs + 1e-15)))

def mcnemar_exact(gold, a, b):
    gold, a, b = map(np.asarray, [gold, a, b])
    ca, cb = (a == gold), (b == gold)
    n10 = int(np.sum(ca & ~cb))  # A correct, B wrong
    n01 = int(np.sum(~ca & cb))  # A wrong, B correct
    d = n10 + n01
    p = binom_exact_two_sided(min(n10, n01), d) if d else 1.0
    return n10, n01, d, p

def mf1(gold, pred):
    return f1_score(gold, pred, labels=LABELS, average="macro", zero_division=0)

def paired_bootstrap(gold, a, b, reps=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    gold, a, b = map(np.asarray, [gold, a, b])
    n = len(gold)
    f1a, f1b = mf1(gold,a), mf1(gold,b)
    rng = np.random.default_rng(seed)
    deltas = np.empty(reps)
    for i in range(reps):
        idx = rng.integers(0, n, size=n)
        deltas[i] = mf1(gold[idx], a[idx]) - mf1(gold[idx], b[idx])
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    return f1a, f1b, f1a-f1b, float(lo), float(hi)

meta = {
    "SP500_RoBERTa_vs_FTFinBERT": ("S&P500","RoBERTa","Fine-tuned FinBERT"),
    "SP500_OriginalFinBERT_vs_FTFinBERT": ("S&P500","Original FinBERT","Fine-tuned FinBERT"),
    "Reuters_OriginalFinBERT_vs_FTFinBERT": ("Reuters","Original FinBERT","Fine-tuned FinBERT"),
    "Reuters_RoBERTa_vs_FTFinBERT": ("Reuters","RoBERTa","Fine-tuned FinBERT"),
}

if not pairs:
    raise ValueError("No paired comparisons could be built. Check selected prediction files above.")

rows = []
for cid, df in pairs.items():
    ds, ma, mb = meta[cid]
    n10,n01,disc,p = mcnemar_exact(df["gold"],df["pred_a"],df["pred_b"])
    f1a,f1b,delta,lo,hi = paired_bootstrap(df["gold"],df["pred_a"],df["pred_b"])
    rows.append({
        "comparison_id":cid,"dataset":ds,"model_a":ma,"model_b":mb,"n":len(df),
        "f1_a":f1a,"f1_b":f1b,"delta_f1_a_minus_b":delta,
        "a_correct_b_wrong":n10,"a_wrong_b_correct":n01,
        "discordant_total":disc,"mcnemar_p_raw":p,
        "bootstrap_ci_low":lo,"bootstrap_ci_high":hi,
        "bootstrap_repetitions":N_BOOTSTRAP
    })

results = pd.DataFrame(rows)
display(results.round(6))


In [ ]:

def holm(pvals):
    p = np.asarray(pvals, float)
    m = len(p)
    order = np.argsort(p)
    out = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        val = min(1.0, (m-rank) * p[idx])
        running = max(running, val)
        out[idx] = running
    return out

results["mcnemar_p_holm"] = holm(results["mcnemar_p_raw"])
results["mcnemar_significant_holm_005"] = results["mcnemar_p_holm"] < 0.05
results["bootstrap_ci_excludes_zero"] = (
    (results["bootstrap_ci_low"] > 0) | (results["bootstrap_ci_high"] < 0)
)

cols = [
    "dataset","model_a","model_b","n","f1_a","f1_b","delta_f1_a_minus_b",
    "mcnemar_p_raw","mcnemar_p_holm","mcnemar_significant_holm_005",
    "bootstrap_ci_low","bootstrap_ci_high","bootstrap_ci_excludes_zero"
]
display(results[cols].round(6))


In [ ]:

def ptxt(p):
    return "<0.000001" if p < 1e-6 else f"{p:.6f}"

texts = []
print("\nTEZDE KULLANILABİLECEK ÖZET\n" + "="*100)

for _, r in results.iterrows():
    if r["delta_f1_a_minus_b"] > 0:
        direction = f"{r['model_a']} lehine"
    elif r["delta_f1_a_minus_b"] < 0:
        direction = f"{r['model_b']} lehine"
    else:
        direction = "fark yok"

    both = bool(r["mcnemar_significant_holm_005"] and r["bootstrap_ci_excludes_zero"])
    if both:
        evidence = "McNemar-Holm ve bootstrap güven aralığı birlikte farkı desteklemektedir."
    else:
        evidence = "İki testin sonuçları birlikte temkinli yorumlanmalıdır."

    text = (
        f"{r['dataset']} | {r['model_a']} vs {r['model_b']}: "
        f"Macro-F1 {r['f1_a']:.4f} vs {r['f1_b']:.4f}; "
        f"ΔF1={r['delta_f1_a_minus_b']:+.4f}; "
        f"Holm-düzeltilmiş McNemar p={ptxt(r['mcnemar_p_holm'])}; "
        f"paired bootstrap %95 GA [{r['bootstrap_ci_low']:+.4f}, {r['bootstrap_ci_high']:+.4f}]. "
        f"{direction}. {evidence}"
    )
    print("\n" + text)
    texts.append({"dataset":r["dataset"],"comparison":f"{r['model_a']} vs {r['model_b']}","thesis_text":text})

thesis_text_df = pd.DataFrame(texts)


In [ ]:
results_path = RESULTS_DIR / "external_statistical_significance_results.csv"
text_path = RESULTS_DIR / "external_statistical_significance_thesis_text.csv"
json_path = RESULTS_DIR / "external_statistical_significance_results.json"
skipped_path = RESULTS_DIR / "external_statistical_significance_skipped_comparisons.csv"

results.to_csv(results_path, index=False, encoding="utf-8-sig")
thesis_text_df.to_csv(text_path, index=False, encoding="utf-8-sig")

skipped_df = pd.DataFrame(skipped_comparisons + missing_inputs)
skipped_df.to_csv(skipped_path, index=False, encoding="utf-8-sig")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results.to_dict(orient="records"), f, ensure_ascii=False, indent=2, default=str)

print("Kaydedildi:")
print(results_path)
print(text_path)
print(json_path)
print(skipped_path)


## Bana göndermen gereken dosya

Çalıştırdıktan sonra şunu yükle:

`results/external_statistical_significance/external_statistical_significance_results.csv`

Bunu aldıktan sonra tezde hangi dış-test farklarının istatistiksel olarak desteklendiğini doğrudan güncelleyebiliriz.
